# F1 Winner Prediction - Full Pipeline (Single Notebook)

Ejecuta el pipeline completo en Colab **sin Google Drive**:
1. Clona el repo de GitHub + instala dependencias
2. Descarga datos de FastF1 (2014-2025)
3. Construye features y secuencias de entrenamiento
4. Entrena el Transformer dual-stream (2014-2024 combinado)
5. Predice la temporada 2025 y compara con resultados reales

**Resultado esperado**: ~54.5% val accuracy, ~41.7% test (10/24 aciertos)

**Tiempo total**: ~45-75 min (mayor parte en descarga de datos)

---
## Celda 1: Clonar Repo + Instalar Dependencias

In [ ]:
# @title 1. Clone Repo & Install Dependencies
!git clone https://github.com/fliupa/f1_transformer.git
%cd f1_transformer
!pip install -q -r requirements.txt

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import time
import os

# NO usamos google.colab.drive - todo en disco local del runtime
# NO seteamos COLAB=1 - usamos paths locales del repo

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    print('No GPU detected - usando CPU (el entrenamiento sera lento)')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()
print(f'Device: {DEVICE}, AMP: {USE_AMP}')

---
## Celda 2: Descargar Datos (FastF1 API)

Descarga resultados de carreras 2014-2025 con rate limiting.
**Tiempo estimado**: 30-60 min para las 12 temporadas.

Si ya ejecutaste esta celda antes, el script detecta datos existentes y solo descarga lo faltante.

In [ ]:
# @title 2. Fetch Race Data from FastF1
from scripts.fetch_data import main as fetch_main

print('=' * 60)
print('DESCARGANDO DATOS DE FastF1 (2014-2025)')
print('Esto puede tomar 30-60 min. Se paciente.')
print('Los datos se guardan en data/raw/races/')
print('=' * 60)

fetch_main()

---
## Celda 3: Base de Datos de Circuitos + Clima

Crea la base de datos estatica de circuitos y descarga clima historico de Open-Meteo.

In [ ]:
# @title 3. Circuit Database & Weather
RAW = Path('data/raw')

# --- Circuit Database ---
circuits_data = [
    ('Albert Park', 5.278, 16, 4, 5, 2, 2, 2, 2, 79.8),
    ('Bahrain', 5.412, 15, 3, 5, 3, 1, 3, 2, 90.9),
    ('Baku', 6.003, 20, 2, -28, 1, 1, 2, 1, 100.3),
    ('Barcelona', 4.675, 16, 2, 100, 3, 2, 2, 1, 76.0),
    ('Hungaroring', 4.381, 14, 2, 150, 3, 3, 2, 1, 76.2),
    ('Interlagos', 4.309, 15, 2, 780, 3, 2, 2, 2, 70.5),
    ('Imola', 4.909, 19, 2, 50, 3, 2, 2, 1, 75.4),
    ('Jeddah', 6.174, 27, 3, 5, 1, 1, 1, 3, 87.4),
    ('Las Vegas', 6.201, 17, 2, 600, 1, 1, 2, 2, 94.0),
    ('Losail', 5.38, 16, 2, 5, 3, 2, 3, 2, 83.6),
    ('Marina Bay', 4.94, 23, 3, 5, 1, 3, 2, 1, 95.2),
    ('Mexico City', 4.304, 17, 3, 2240, 3, 2, 1, 2, 77.3),
    ('Miami', 5.412, 19, 3, 2, 2, 2, 2, 2, 88.5),
    ('Monaco', 3.337, 19, 2, 5, 1, 3, 2, 1, 70.4),
    ('Monza', 5.793, 11, 2, 160, 3, 1, 2, 3, 79.3),
    ('Montreal', 4.361, 14, 3, 5, 2, 1, 2, 2, 73.0),
    ('Mugello', 5.245, 15, 2, 250, 3, 2, 2, 1, 78.5),
    ('Paul Ricard', 5.842, 15, 2, 400, 3, 2, 2, 2, 88.5),
    ('Portimao', 4.653, 15, 2, 100, 3, 2, 2, 2, 77.4),
    ('Red Bull Ring', 4.318, 10, 3, 700, 3, 1, 2, 2, 65.0),
    ('Sepang', 5.543, 15, 2, 50, 3, 2, 3, 2, 94.2),
    ('Shanghai', 5.451, 16, 2, 5, 3, 2, 3, 2, 92.0),
    ('Silverstone', 5.891, 18, 3, 150, 3, 2, 2, 3, 87.1),
    ('Sochi', 5.848, 18, 2, 5, 2, 2, 2, 2, 90.8),
    ('Spa', 7.004, 19, 2, 450, 3, 1, 2, 3, 101.9),
    ('Suzuka', 5.807, 18, 2, 100, 3, 3, 2, 2, 89.2),
    ('Yas Marina', 5.281, 21, 2, 5, 3, 2, 2, 2, 85.2),
    ('Zandvoort', 4.259, 14, 2, 5, 3, 3, 2, 1, 71.3),
    ('Austin', 5.513, 20, 2, 170, 3, 2, 2, 2, 96.0),
    ('Hockenheim', 4.574, 17, 2, 100, 3, 2, 2, 2, 73.7),
    ('Nurburgring', 5.148, 15, 2, 600, 3, 2, 2, 2, 87.1),
    ('Istanbul', 5.338, 14, 2, 57, 3, 2, 2, 2, 84.7),
    ('Monte Carlo', 3.337, 19, 2, 5, 1, 3, 2, 1, 70.4),
]
cols = ['circuit_name','length_km','corners','drs_zones','altitude_m',
        'track_type_code','downforce_code','tyre_degradation_code',
        'overtaking_code','lap_record_s']

(RAW / 'circuits').mkdir(parents=True, exist_ok=True)
circuits_df = pd.DataFrame([dict(zip(cols, c)) for c in circuits_data])
circuits_df.to_csv(RAW / 'circuits' / 'circuits.csv', index=False)
print(f'Circuits: {len(circuits_df)} circuitos guardados')

# --- Weather from Open-Meteo ---
import requests

races_df = pd.read_csv(RAW / 'races' / 'race_results_all.csv')
races_df['event_date'] = pd.to_datetime(races_df['event_date'], format='mixed')
unique_races = races_df[['year','round','circuit','event_date']].drop_duplicates()

coords = {
    'Albert Park': (-37.85,144.97), 'Bahrain': (26.03,50.51),
    'Baku': (40.37,49.85), 'Barcelona': (41.57,2.26),
    'Hungaroring': (47.58,19.25), 'Interlagos': (-23.70,-46.70),
    'Imola': (44.34,11.71), 'Jeddah': (21.63,39.10),
    'Las Vegas': (36.11,-115.17), 'Losail': (25.49,51.45),
    'Marina Bay': (1.29,103.86), 'Mexico City': (19.40,-99.09),
    'Miami': (25.96,-80.24), 'Monaco': (43.73,7.42),
    'Monza': (45.62,9.28), 'Montreal': (45.50,-73.52),
    'Mugello': (43.99,11.37), 'Nurburgring': (50.33,6.95),
    'Paul Ricard': (43.25,5.79), 'Portimao': (37.23,-8.64),
    'Red Bull Ring': (47.22,14.76), 'Sepang': (2.76,101.74),
    'Shanghai': (31.34,121.22), 'Silverstone': (52.07,-1.02),
    'Sochi': (43.41,39.97), 'Spa': (50.44,5.97),
    'Suzuka': (34.84,136.54), 'Yas Marina': (24.47,54.60),
    'Zandvoort': (52.39,4.54), 'Austin': (30.13,-97.64),
    'Hockenheim': (49.33,8.57), 'Istanbul': (40.95,29.40),
    'Monte Carlo': (43.73,7.42),
}

BASE_URL = 'https://archive-api.open-meteo.com/v1/archive'
(RAW / 'weather').mkdir(parents=True, exist_ok=True)
weather_records = []

print(f'Fetching weather for {len(unique_races)} races...')
for _, race in tqdm(unique_races.iterrows(), total=len(unique_races)):
    circuit = str(race['circuit'])
    coord = None
    for name, c in coords.items():
        if name.lower() in circuit.lower() or circuit.lower() in name.lower():
            coord = c; break
    if coord is None: continue
    date = race['event_date']
    if pd.isna(date): continue
    params = {
        'latitude': coord[0], 'longitude': coord[1],
        'start_date': date.strftime('%Y-%m-%d'),
        'end_date': date.strftime('%Y-%m-%d'),
        'daily': 'temperature_2m_mean,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_mean,surface_pressure_mean',
        'timezone': 'auto'
    }
    try:
        resp = requests.get(BASE_URL, params=params, timeout=10)
        data = resp.json()
        if 'daily' in data and data['daily']['temperature_2m_mean']:
            d = data['daily']
            weather_records.append({
                'year': int(race['year']), 'round': int(race['round']),
                'circuit': circuit, 'event_date': date,
                'openmeteo_temp_mean': d['temperature_2m_mean'][0],
                'openmeteo_humidity': d['relative_humidity_2m_mean'][0],
                'openmeteo_precip_mm': d['precipitation_sum'][0],
                'openmeteo_wind_speed': d['wind_speed_10m_mean'][0],
                'openmeteo_pressure': d['surface_pressure_mean'][0],
                'openmeteo_rain_flag': 1 if d['precipitation_sum'][0] > 0.5 else 0,
            })
        time.sleep(0.15)
    except: continue

weather_df = pd.DataFrame(weather_records)
weather_df.to_csv(RAW / 'weather' / 'weather_all.csv', index=False)
print(f'Weather: {len(weather_df)} registros guardados')

---
## Celda 4: Construir Dataset

Ejecuta el pipeline de feature engineering:
- Calcula form reciente de pilotos/constructores
- Normaliza features numericas
- Construye secuencias (context=10 carreras previas, 20 pilotos por carrera)
- Crea encoders (driver, constructor, circuit)
- Guarda features_train.pt (192 seqs), features_val.pt (33 seqs), features_2025.pt, metadata.pkl

In [ ]:
# @title 4. Build Dataset (Feature Engineering + Sequences)
from src.preprocessing.build_dataset import main as build_main

print('=' * 60)
print('CONSTRUYENDO DATASET')
print('=' * 60)

build_main()

# Verificar
PROCESSED = Path('data/processed')
train_data = torch.load(PROCESSED / 'features_train.pt', weights_only=False)
val_data = torch.load(PROCESSED / 'features_val.pt', weights_only=False)
data_2025 = torch.load(PROCESSED / 'features_2025.pt', weights_only=False)

with open(PROCESSED / 'metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

print(f'\nTrain: {len(train_data["winners"])} seqs')
print(f'Val:   {len(val_data["winners"])} seqs')
print(f'2025:  {len(data_2025["winners"])} seqs')
print(f'Context window: {metadata["context_window"]} races')
print(f'Drivers: {metadata["num_drivers"]}, Constructors: {metadata["num_constructors"]}, Circuits: {metadata["num_circuits"]}')
print(f'Features - Candidate: {metadata["d_candidate_raw"]}, Context: {metadata["d_context_raw"]}')

---
## Celda 5: Entrenar Modelo

Transformer dual-stream con cross-attention:
- 2.28M parametros, d_model=192, 6 heads, 3 encoder layers
- 80 epochs con early stopping (patience=15)
- **Tiempo estimado**: 8-12 min en GPU T4, 30-45 min en CPU
- **Target**: ~54.5% val accuracy

In [ ]:
# @title 5. Train Model (2014-2024 Combined)
from src.model.transformer_model import F1WinnerTransformer
from src.training.trainer import Trainer

MODEL_DIR = Path('models/final')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_data = torch.load(PROCESSED / 'features_train.pt', weights_only=False)
val_data = torch.load(PROCESSED / 'features_val.pt', weights_only=False)
with open(PROCESSED / 'metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

model = F1WinnerTransformer(
    d_model=192, n_heads=6, n_encoder_layers=3, n_cross_attn_layers=2,
    d_ff=768, dropout=0.15,
    context_window=metadata['context_window'],
    num_drivers=metadata['num_drivers'],
    num_constructors=metadata['num_constructors'],
    num_circuits=metadata['num_circuits'],
    d_candidate_raw=metadata['d_candidate_raw'],
    d_context_raw=metadata['d_context_raw'],
)

params = sum(p.numel() for p in model.parameters())
print(f'Model: {params:,} params')
print(f'Train: {len(train_data["winners"])} seqs, Val: {len(val_data["winners"])} seqs')
print(f'Device: {DEVICE}, AMP: {USE_AMP}')
print()

trainer = Trainer(
    model=model,
    train_data=train_data,
    val_data=val_data,
    device=DEVICE,
    batch_size=32,
    learning_rate=5e-4,
    weight_decay=1e-5,
    epochs=80,
    patience=15,
    use_amp=USE_AMP,
    checkpoint_dir=MODEL_DIR,
    num_workers=2 if torch.cuda.is_available() else 0,
)

history = trainer.train(early_stopping=True)
print(f'\nBest val acc: {trainer.best_val_acc:.3f} (epoch {trainer.best_epoch+1})')

---
## Celda 6: Curvas de Entrenamiento

In [ ]:
# @title 6. Training Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss (2014-2024 Combined)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['val_acc'], label='Val Accuracy', linewidth=2, color='green')
axes[1].axhline(y=0.20, color='gray', ls='--', alpha=0.5, label='Champion Leader (~20%)')
axes[1].axhline(y=0.05, color='red', ls='--', alpha=0.5, label='Random (5%)')
axes[1].axhline(y=0.40, color='orange', ls='--', alpha=0.5, label='Pole Position (~38-42%)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f'\nBest val accuracy: {trainer.best_val_acc*100:.1f}%')
print(f'2.7x better than champion leader baseline (20%)')

---
## Celda 7: Predecir Temporada 2025

Carga el modelo entrenado y predice el ganador de las 24 carreras.
Compara contra los resultados reales.

In [ ]:
# @title 7. Predict 2025 Season & Compare vs Real Results
from src.model.transformer_model import F1WinnerTransformer

# Resultados reales 2025
REAL_WINNERS = {
    1: 'NOR', 2: 'PIA', 3: 'VER', 4: 'RUS', 5: 'VER', 6: 'NOR',
    7: 'PIA', 8: 'NOR', 9: 'VER', 10: 'NOR', 11: 'PIA', 12: 'RUS',
    13: 'PIA', 14: 'VER', 15: 'VER', 16: 'NOR', 17: 'PIA', 18: 'VER',
    19: 'PIA', 20: 'VER', 21: 'PIA', 22: 'VER', 23: 'NOR', 24: 'NOR',
}

# Cargar modelo entrenado
ckpt = torch.load(MODEL_DIR / 'best.pt', map_location='cpu', weights_only=False)
model = F1WinnerTransformer(
    d_model=192, n_heads=6, n_encoder_layers=3, n_cross_attn_layers=2,
    d_ff=768, dropout=0.15,
    context_window=metadata['context_window'],
    num_drivers=metadata['num_drivers'],
    num_constructors=metadata['num_constructors'],
    num_circuits=metadata['num_circuits'],
    d_candidate_raw=metadata['d_candidate_raw'],
    d_context_raw=metadata['d_context_raw'],
)
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(DEVICE)
model.eval()

data_2025 = torch.load(PROCESSED / 'features_2025.pt', weights_only=False)
driver_enc = metadata['driver_encoder']

ctx, cand, gaps = data_2025['context'], data_2025['candidates'], data_2025['time_gaps']
all_preds = []; summary = []; correct = 0

print('Prediciendo 24 carreras...')
with torch.no_grad():
    for i in tqdm(range(len(ctx))):
        logits = model(ctx[i:i+1].to(DEVICE), cand[i:i+1].to(DEVICE), gaps[i:i+1].to(DEVICE))
        probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
        driver_idx = cand[i, :, 0].long().numpy()
        names = [driver_enc.decode(int(idx)) for idx in driver_idx]
        si = np.argsort(probs)[::-1]
        pred = names[si[0]]
        rnd = i + 1
        real = REAL_WINNERS.get(rnd, '?')
        match = pred == real
        if match:
            correct += 1
        summary.append({
            'round': rnd, 'predicted': pred, 'actual': real,
            'correct': match, 'confidence': float(probs[si[0]]),
            'top3': [names[idx] for idx in si[:3]]
        })
        for rank, idx in enumerate(si):
            nm = names[idx]
            if not nm.startswith('UNK'):
                all_preds.append({'year': 2025, 'round': rnd, 'rank': rank+1,
                                  'driver': nm, 'win_probability': float(probs[idx])})

pred_df = pd.DataFrame(all_preds)
summary_df = pd.DataFrame(summary)
print(f'\nAccuracy: {correct}/{len(ctx)} = {correct/len(ctx)*100:.1f}%')

---
## Celda 8: Resultados - Predicciones vs Realidad

In [ ]:
# @title 8. Predicted vs Real Winners
print('=' * 70)
print(f'{"R":>3s} {"Pred":<6s} {"Conf":>6s} {"Real":<6s} {"Match":>6s}  Top 3')
print('-' * 70)
for _, row in summary_df.iterrows():
    m = 'YES' if row['correct'] else 'NO'
    print(f'{int(row["round"]):3d} {row["predicted"]:<6s} {row["confidence"]:6.1%} '
          f'{row["actual"]:<6s} {m:>6s}  {row["top3"]}')
print('-' * 70)
print(f'\nTOTAL: {correct}/24 = {correct/24*100:.1f}%')

---
## Celda 9: Visualizaciones

In [ ]:
# @title 9. Championship Projection vs Reality
winners = pred_df[pred_df['rank'] == 1].sort_values('round')
pred_wins = winners['driver'].value_counts()
real_wins = pd.Series(REAL_WINNERS).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Predicted
bars1 = axes[0].bar(pred_wins.index, pred_wins.values,
                    color=sns.color_palette('husl', len(pred_wins)))
axes[0].set_title('Predicted 2025 Wins', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Wins'); axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(pred_wins.values):
    axes[0].text(i, v+0.2, str(v), ha='center', fontweight='bold')

# Real
colors = ['#1E41FF' if d == 'VER' else '#FF8700' if d in ('NOR', 'PIA')
          else '#00D2BE' if d == 'RUS' else 'gray' for d in real_wins.index]
axes[1].bar(real_wins.index, real_wins.values, color=colors)
axes[1].set_title('Real 2025 Wins', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Wins'); axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(real_wins.values):
    axes[1].text(i, v+0.2, str(v), ha='center', fontweight='bold')

plt.tight_layout(); plt.show()

# Comparison table
print(f'\n{"Driver":<8s} {"Pred":>5s} {"Real":>5s}')
print('-' * 20)
for d in sorted(set(list(pred_wins.index) + list(real_wins.index)),
                key=lambda x: real_wins.get(x, 0), reverse=True):
    print(f'{d:<8s} {pred_wins.get(d, 0):>5d} {real_wins.get(d, 0):>5d}')

---
## Celda 10: Analisis Final

In [ ]:
# @title 10. Analysis & Insights
print('=' * 65)
print('F1 2025 SEASON PREDICTION - ANALYSIS')
print('=' * 65)
print()

print(f'Correct predictions ({correct}/24 = {correct/24*100:.1f}%):')
for _, row in summary_df[summary_df['correct']].iterrows():
    print(f'  R{int(row["round"]):2d}: {row["predicted"]} (conf: {row["confidence"]:.1%})')

print(f'\nIncorrect predictions ({24-correct}/24):')
for _, row in summary_df[~summary_df['correct']].iterrows():
    print(f'  R{int(row["round"]):2d}: Pred={row["predicted"]} | Real={row["actual"]} '
          f'(conf: {row["confidence"]:.1%})')

print()
print('Key observations:')
print('  1. VER tiende a ser sobre-predicho (sesgo de dominancia 2023)')
print('  2. NOR identificado correctamente como contendiente top')
print('  3. PIA sub-estimado - temporada breakout no anticipada')
print('  4. RUS sin victorias predichas (2 reales)')
print('  5. HAM sobre-estimado por hype del cambio a Ferrari')

correct_conf = summary_df[summary_df['correct']]['confidence'].mean()
incorrect_conf = summary_df[~summary_df['correct']]['confidence'].mean()
print(f'\nConfianza promedio:')
print(f'  Predicciones correctas: {correct_conf:.1%}')
print(f'  Predicciones incorrectas: {incorrect_conf:.1%}')
print(f'  (El modelo muestra igual confianza en aciertos y errores)')

print(f'\n{"="*65}')
print(f'Baselines de comparacion:')
print(f'  Random guess:       5.0%')
print(f'  Champion leader:   ~20.0%')
print(f'  Pole position:     ~38-42%')
print(f'  TRANSFORMER:       {correct/24*100:.1f}%')
print(f'{"="*65}')
print()
print('PIPELINE COMPLETO EJECUTADO CON EXITO!')
print('Todo en disco local de Colab, sin Google Drive.')